# E10-3: Multi-Resolution Sinusoidal Model

Extend the sinusoidal model to use **multiple analysis windows of different sizes** applied to different frequency bands — a technique called *multi-resolution analysis*.

**Key concepts:**
- **Time–frequency tradeoff:** long windows → better frequency resolution; short windows → better time resolution
- **Band-partitioned analysis:** split the spectrum into bands B1, B2, B3 and apply a dedicated window and FFT size to each band
- **`sineModelMultiRes()`:** takes three windows `w1, w2, w3`, three FFT sizes `N1, N2, N3`, and three frequency band boundaries `B`
- **Target material:** polyphonic recordings with both melodic (low-frequency, slow-varying) and percussive (broadband, transient) content

**Workflow:**
- **Part 1** — Implement `sineModelMultiRes()` by adapting `sineModel()`
- **Part 2** — Analyse Sound 1: compare baseline `sineModel()` against `sineModelMultiRes()`
- **Part 3** — Analyse Sound 2: compare baseline `sineModel()` against `sineModelMultiRes()`


In [ ]:
import sys, os
import math
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import get_window
from scipy.fftpack import ifft, fftshift
from smstools import dftModel as DFT
from smstools import utilFunctions as UF
from smstools import sineModel as SM
import IPython.display as ipd


## Part 1 — Implement `sineModelMultiRes()`

Adapt `sineModel()` to support three parallel DFT analyses, one per frequency band.

**Steps:**
1. Copy the frame-loop body from `sineModel()` into `sineModelMultiRes()`
2. Inside each frame, compute three DFTs using `dftAnal()` with `(x_frame, w1, N1)`, `(x_frame, w2, N2)`, `(x_frame, w3, N3)` — note that each window has a different size so the zero-padded frame must be adjusted accordingly
3. Restrict peak-picking for each spectrum to its assigned band: peaks of `X1` where `f < B[0]`; peaks of `X2` where `B[0] ≤ f < B[1]`; peaks of `X3` where `f ≥ B[1]`
4. Merge the three peak lists (frequencies, magnitudes, phases) and synthesise the frame with `dftSynth()`


In [ ]:
# E10-3.1: implement sineModelMultiRes() by adapting sineModel()
def sineModelMultiRes(x, fs, w, N, t, B):
    """
    Analysis/synthesis of a sound using a multi-resolution sinusoidal model.
    Inputs:
        x: input array sound
        fs: sampling rate
        w: list of 3 analysis windows [w1, w2, w3] with decreasing sizes
        N: list of 3 FFT sizes [N1, N2, N3] (power of 2, >= corresponding window size)
        t: peak-picking threshold in negative dB
        B: list of 2 frequency band boundaries in Hz, e.g. [1000, 5000]
           band 1: 0 <= f < B[0]  → analysed with w[0], N[0]
           band 2: B[0] <= f < B[1] → analysed with w[1], N[1]
           band 3: f >= B[1]      → analysed with w[2], N[2]
    Output:
        y: output array sound
    """

    ### your code here — start from copying the frame-loop of sineModel()



### E10-3.2 — Implementation design questions

Answer the following questions **after** completing `sineModelMultiRes()`.

1. **Window sizes and FFT sizes.** The suggested sizes are `M1=4095, M2=2047, M3=1023`. Why must each FFT size `N` be the next power of two *greater than or equal to* the window size? What are the corresponding `N1`, `N2`, `N3`? How does zero-padding affect the frequency resolution visible in the spectrum?

2. **Frame alignment.** `sineModel()` uses a single hop size `H = M//4`. When three windows have different sizes, which window determines the hop size, and why? What happens to the time-domain reconstruction if you chose the hop size of the *smallest* window instead?

3. **Band boundary choice.** The suggested boundaries are `B = [1000, 5000]` Hz. Describe a genre of music (e.g. piano solo, drum kit, string quartet) where you would shift B[0] *lower* and explain the perceptual reason. What artefact would you hear at the boundary if the transition between bands is not handled correctly?

4. **Peak merging.** After peak-picking in each band, you merge three lists of `(freqsMag, phaseMag)`. Could two bands ever detect the same sinusoidal component? Under what conditions (e.g. a low-frequency partial whose sidelobes leak into B2) might this happen, and how would you prevent double-counting?

5. **Computational cost.** Count the number of DFT bins computed per frame for the single-window baseline (window size `M=4095`, `N=4096`) vs the three-window version. Is `sineModelMultiRes()` more or less expensive? State your result as a ratio and explain whether the cost is justified by the quality improvement.


## Part 2 — Baseline vs multi-resolution: Sound 1

Choose a polyphonic recording from Freesound with **both melodic and percussive content** (e.g. piano with drums, guitar with claps). Convert it to mono WAV, trim to ≤10 s.

**Steps:**
1. Set parameters for `sineModel()` (single window, `E10-3.3`) — tune `M`, `N`, `t`, `window` to get the best possible reconstruction
2. Run `sineModelMultiRes()` with the same threshold but three window sizes and band boundaries (`E10-3.4`) — tune `M`, `N`, `B` for best result
3. Listen to original, baseline output, and multi-resolution output; observe spectrograms


In [ ]:
# E10-3.3: baseline sinusoidal analysis/synthesis — Sound 1
### set the parameters
input_file = 'XXX'   # path to Sound 1 (mono WAV)
M = XX               # window size (odd, e.g. 4095 for good low-freq resolution)
N = XX               # FFT size (power of 2 >= M, e.g. 4096)
t = XX               # peak-picking threshold in negative dB (e.g. -80)
window = 'XX'        # window type, e.g. 'blackman'

# no need to change code from here
fs, x = UF.wavread(input_file)
w = get_window(window, M)
y_baseline = SM.sineModel(x, fs, w, N, t)

print(f'Input:    {len(x)/fs:.2f} s  |  Output: {len(y_baseline)/fs:.2f} s')
ipd.display(ipd.Audio(data=x, rate=fs))
ipd.display(ipd.Audio(data=y_baseline, rate=fs))

# visualise spectrogram comparison
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.specgram(x, NFFT=N, Fs=fs, noverlap=N//2, cmap='inferno')
plt.title('Original — Sound 1'); plt.xlabel('Time (s)'); plt.ylabel('Frequency (Hz)')
plt.subplot(1, 2, 2)
plt.specgram(y_baseline, NFFT=N, Fs=fs, noverlap=N//2, cmap='inferno')
plt.title('Baseline sineModel — Sound 1'); plt.xlabel('Time (s)'); plt.ylabel('Frequency (Hz)')
plt.tight_layout()
plt.show()


In [ ]:
# E10-3.4: multi-resolution sinusoidal analysis/synthesis — Sound 1
### set the parameters (tune these for best result)
M_list   = [XX, XX, XX]    # window sizes for bands 1, 2, 3 (e.g. [4095, 2047, 1023])
N_list   = [XX, XX, XX]    # FFT sizes (power of 2 >= each M, e.g. [4096, 2048, 1024])
t        = XX              # peak-picking threshold in negative dB (same as baseline)
B        = [XX, XX]        # frequency band boundaries in Hz (e.g. [1000, 5000])
window   = 'XX'            # window type (e.g. 'blackman')

# no need to change code from here
w_list = [get_window(window, m) for m in M_list]
y_multi = sineModelMultiRes(x, fs, w_list, N_list, t, B)

print(f'Multi-res output: {len(y_multi)/fs:.2f} s')
ipd.display(ipd.Audio(data=y_multi, rate=fs))

# visualise spectrogram comparison
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.specgram(y_baseline, NFFT=N_list[0], Fs=fs, noverlap=N_list[0]//2, cmap='inferno')
plt.title('Baseline sineModel — Sound 1'); plt.xlabel('Time (s)'); plt.ylabel('Frequency (Hz)')
plt.subplot(1, 2, 2)
plt.specgram(y_multi, NFFT=N_list[0], Fs=fs, noverlap=N_list[0]//2, cmap='inferno')
plt.title('sineModelMultiRes — Sound 1'); plt.xlabel('Time (s)'); plt.ylabel('Frequency (Hz)')
plt.tight_layout()
plt.show()


### E10-3.5 — Sound 1 analysis questions

Answer based on your own parameter choices and the output you hear/see.

1. **Freesound URL and justification.** Paste the Freesound link to Sound 1. Why is it a good test case for multi-resolution analysis? Name the specific melodic and percussive elements present in the recording.

2. **Parameter table.** Fill in the table for the baseline and multi-resolution runs:

   | Model | `M` (or `M1/M2/M3`) | `N` (or `N1/N2/N3`) | `t` | Band boundaries `B` | Window type |
   |---|---|---|---|---|---|
   | Baseline `sineModel` | | | | — | |
   | `sineModelMultiRes` | | | | | |

3. **Onset sharpness.** Listen to a percussive transient (e.g. a drum hit or piano attack) in the baseline output and the multi-resolution output. Describe the difference in sharpness. In the spectrograms, identify the time frames around the onset and comment on the width of the onset smearing in each model (estimate in milliseconds).

4. **Low-frequency resolution.** Identify a melodic passage with two closely-spaced notes (e.g. a minor second or whole tone). Does `sineModelMultiRes` resolve both pitches more cleanly than the baseline? If yes, explain which band (B1, B2, or B3) is responsible. If no, propose a different `B[0]` or `M1` that might help.

5. **Artefacts.** Describe any audible artefact or unwanted sound introduced by `sineModelMultiRes` that is *not* present in the baseline. At which frequency range does it occur? What parameter change would you try to reduce it?


## Part 3 — Baseline vs multi-resolution: Sound 2

Choose a **second** polyphonic recording from Freesound, different in character from Sound 1 (e.g. if Sound 1 was piano+drums, choose guitar+percussion or strings+plucked bass).

**Steps:**
1. Run the same baseline analysis (`E10-3.6`) and multi-resolution analysis (`E10-3.7`) on Sound 2
2. Tune parameters independently — the optimal `B` and window sizes may differ from Sound 1
3. Compare results across both sounds to draw general conclusions about multi-resolution analysis


In [ ]:
# E10-3.6: baseline sinusoidal analysis/synthesis — Sound 2
### set the parameters
input_file2 = 'XXX'   # path to Sound 2 (mono WAV, different from Sound 1)
M2   = XX             # window size
N2   = XX             # FFT size (power of 2 >= M2)
t2   = XX             # peak-picking threshold in negative dB
window2 = 'XX'        # window type

# no need to change code from here
fs2, x2 = UF.wavread(input_file2)
w2 = get_window(window2, M2)
y_baseline2 = SM.sineModel(x2, fs2, w2, N2, t2)

print(f'Input:    {len(x2)/fs2:.2f} s  |  Output: {len(y_baseline2)/fs2:.2f} s')
ipd.display(ipd.Audio(data=x2, rate=fs2))
ipd.display(ipd.Audio(data=y_baseline2, rate=fs2))

# visualise spectrogram comparison
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.specgram(x2, NFFT=N2, Fs=fs2, noverlap=N2//2, cmap='inferno')
plt.title('Original — Sound 2'); plt.xlabel('Time (s)'); plt.ylabel('Frequency (Hz)')
plt.subplot(1, 2, 2)
plt.specgram(y_baseline2, NFFT=N2, Fs=fs2, noverlap=N2//2, cmap='inferno')
plt.title('Baseline sineModel — Sound 2'); plt.xlabel('Time (s)'); plt.ylabel('Frequency (Hz)')
plt.tight_layout()
plt.show()


In [ ]:
# E10-3.7: multi-resolution sinusoidal analysis/synthesis — Sound 2
### set the parameters (tune independently from Sound 1)
M2_list  = [XX, XX, XX]    # window sizes for bands 1, 2, 3
N2_list  = [XX, XX, XX]    # FFT sizes (power of 2 >= each M)
t2_multi = XX              # peak-picking threshold in negative dB
B2       = [XX, XX]        # frequency band boundaries in Hz
window2  = 'XX'            # window type

# no need to change code from here
w2_list = [get_window(window2, m) for m in M2_list]
y_multi2 = sineModelMultiRes(x2, fs2, w2_list, N2_list, t2_multi, B2)

print(f'Multi-res output: {len(y_multi2)/fs2:.2f} s')
ipd.display(ipd.Audio(data=y_multi2, rate=fs2))

# visualise spectrogram comparison
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.specgram(y_baseline2, NFFT=N2_list[0], Fs=fs2, noverlap=N2_list[0]//2, cmap='inferno')
plt.title('Baseline sineModel — Sound 2'); plt.xlabel('Time (s)'); plt.ylabel('Frequency (Hz)')
plt.subplot(1, 2, 2)
plt.specgram(y_multi2, NFFT=N2_list[0], Fs=fs2, noverlap=N2_list[0]//2, cmap='inferno')
plt.title('sineModelMultiRes — Sound 2'); plt.xlabel('Time (s)'); plt.ylabel('Frequency (Hz)')
plt.tight_layout()
plt.show()


### E10-3.8 — Sound 2 analysis questions

Answer based on your own parameter choices and the output you hear/see.

1. **Freesound URL and justification.** Paste the Freesound link to Sound 2. How does its spectral character differ from Sound 1? Does it have a higher or lower ratio of percussive to melodic content?

2. **Parameter differences.** Compare the `M_list` and `B` values you chose for Sound 2 vs Sound 1. What acoustic feature of Sound 2 motivated any differences? If you used the same parameters, explain why the same settings were appropriate.

3. **Onset vs pitch resolution.** Pick a specific time instant in Sound 2 where a percussive event and a sustained pitch overlap (e.g. a kick drum under a bass guitar). Describe what you hear in the baseline vs multi-resolution output at that moment.

4. **Band boundary appropriateness.** Examine the spectrogram of Sound 2. Are there spectral components that fall exactly at your band boundaries `B[0]` and `B[1]` Hz? If so, do you observe any audible discontinuity or amplitude modulation at those frequencies in the output? How would you adjust the boundaries to avoid this?


### E10-3.9 — Cross-sound comparison and model extension

Answer with reference to both sounds and your experimental results.

1. **Summary table.** Fill in the table for all four analysis runs (baseline and multi-resolution for each sound). Rate the *onset sharpness* and *pitch resolution* on a 1–5 scale, and note any audible artefacts:

   | Run | Sound | Model | `M` / `M1,2,3` | `B` | Onset sharpness (1–5) | Pitch resolution (1–5) | Main artefact |
   |---|---|---|---|---|---|---|---|
   | E10-3.3 | 1 | Baseline | | — | | | |
   | E10-3.4 | 1 | MultiRes | | | | | |
   | E10-3.6 | 2 | Baseline | | — | | | |
   | E10-3.7 | 2 | MultiRes | | | | | |

2. **Computational complexity.** For each of your four runs, count the total number of DFT bins computed *per frame* (use your actual `N` values). Is the multi-resolution version consistently more expensive than the baseline? Under what conditions could it be made cheaper?

3. **Extension to HPR/HPS.** The harmonic + residual (HPR) model requires tracking the fundamental frequency `f0` across frames. Describe two specific challenges that arise when the `f0` of a bass instrument is in band B1 (analysed with a large window `M1`) while the harmonics extend into B2 and B3 (analysed with smaller windows):
   - How does the different time resolution per band affect sinusoid tracking?
   - How does it complicate the separation of the harmonic component from the residual?

4. **Further improvements.** Propose and briefly justify two concrete improvements to `sineModelMultiRes()` beyond what you have implemented. Examples: adaptive band boundaries based on spectral flux, smooth cross-fade at band boundaries, dynamic window sizing per frame. For each proposal, state the expected benefit and a potential drawback.

5. **When to use multi-resolution.** Based on your experiments, give a concrete recommendation: for which types of polyphonic audio content does multi-resolution analysis give the most audible improvement over a single-window baseline, and for which types does it make little or no difference? Justify with specific examples from your own results.
